# Week 2: baseline serving benchmark + capacity math

**Kaggle settings:** Accelerator = *GPU T4 x2*, Internet = *on*. **Secrets** (optional): `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD`.

Launch from the laptop with the Week 1 adapter attached (merged here in about 2 minutes, so no 3.4 GB upload):

`python scripts/kaggle_run.py run --week 2 --attach models/week1_run3/sft/adapter=qwen-sql-week1-adapter`

What runs: the fine-tuned fp16 model served by vLLM on **one** T4 (the TP=2-vs-replicas comparison is Week 3). Then closed-loop concurrency sweeps on 4 frozen workloads, an open-loop Poisson sweep on the Spider workload, and a `vllm bench serve` cross-check. The final cell writes `week2_results.zip`.

In [ ]:
BASE = "Qwen/Qwen3-1.7B"
OUT = "/kaggle/working/week2"
SPIDER = "/kaggle/working/data/spider_data"
MODEL = "/kaggle/working/sft-merged"
URL, SERVED = "http://localhost:8000", "qwen-sql"
WL = f"{OUT}/workloads"
SWEEP = {  # closed-loop concurrency per workload; long_v1 has 100 prompts, so c=64 would be one wave
    "short_v1": [1, 2, 4, 8, 16, 32, 64],
    "long_v1": [1, 2, 4, 8, 16, 32],
    "shared_prefix_v1": [1, 2, 4, 8, 16, 32, 64],
    "spider_v1": [1, 2, 4, 8, 16, 32, 64],
}

import os, subprocess, json
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for k in ["MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"]:
    try:
        os.environ[k] = _s.get_secret(k)
    except Exception:
        print(f"secret {k} not set; that integration stays off")
os.makedirs(OUT, exist_ok=True)
STATUS = {}

def run(step, cmd, needs=()):
    """Run a shell step, stream its output, keep a per-step log, never raise.
    A failed step marks itself so that steps depending on it are skipped, not re-failed."""
    blocked = [n for n in needs if STATUS.get(n) != "ok"]
    if blocked:
        STATUS[step] = "skipped"
        print(f"--- skip {step}: needs {blocked}")
        return False
    print(f"--- {step}: {cmd}")
    with open(f"{OUT}/{step}.log", "w") as log:
        proc = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in proc.stdout:
            log.write(line)
            if "it/s]" not in line and "s/it]" not in line:
                print(line, end="")
    STATUS[step] = "ok" if proc.wait() == 0 else "failed"
    print(f"--- {step}: {STATUS[step]}")
    return STATUS[step] == "ok"

In [ ]:
import glob, shutil, subprocess, tarfile
DEST = "/kaggle/working/repo"
marker = glob.glob("/kaggle/input/**/COMMIT", recursive=True)
if marker:  # code uploaded by scripts/kaggle_run.py
    commit = open(marker[0]).read().strip()
    tars = glob.glob("/kaggle/input/**/src.tar", recursive=True)
    trees = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
    os.makedirs(DEST, exist_ok=True)
    if tars:
        with tarfile.open(tars[0]) as t:
            t.extractall(DEST, filter="data")
    elif trees:  # Kaggle unpacks uploaded archives itself (into src/)
        shutil.copytree(os.path.dirname(trees[0]), DEST, dirs_exist_ok=True)
    else:
        raise RuntimeError("code dataset attached but neither src.tar nor pyproject.toml found")
else:
    !git clone -q {REPO} {DEST}
    commit = subprocess.getoutput(f"git -C {DEST} rev-parse --short HEAD")
open(f"{OUT}/COMMIT", "w").write(commit + "\n")
print("code commit:", commit)
%cd {DEST}
!pip install -q -e . -r requirements-gpu.txt 2>&1 | tail -3
# The image ships torchao 0.10; PEFT 0.20 raises on torchao < 0.16 when loading LoRA onto a plain
# (non-quantized) model, which broke the merge in run 3. Nothing here uses torchao.
!pip uninstall -y -q torchao

# Fail fast: a broken setup must not burn GPU hours on cells that can only fail.
check = subprocess.run(["python", "-c", "import bench.client, bench.server, bench.report, finetune.merge, vllm, peft"],
                       capture_output=True, text=True)
if check.returncode != 0:
    raise RuntimeError("repo or serving deps not importable:\n" + check.stderr[-2000:])
print("repo and serving deps import OK")

## 1. Environment record

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv | tee {OUT}/gpu.txt
!pip freeze | grep -iE "^(vllm|torch|transformers|peft)==" | tee {OUT}/versions.txt
!nproc | tee {OUT}/cpus.txt

## 2. Data and frozen workloads (MANIFEST.json holds each file's sha256)

In [ ]:
!python scripts/get_spider.py --out /kaggle/working/data
run("workloads", f"python -m bench.workloads --tokenizer {BASE} --out {WL} --spider-dir {SPIDER}")

## 3. Rebuild the Week 1 fine-tuned model (merge the attached adapter into the fp16 base)

In [ ]:
import glob
found = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
if found:
    run("merge", f"python -m finetune.merge --base {BASE} --adapter {os.path.dirname(found[0])} --out {MODEL}")
else:
    STATUS["merge"] = "failed"
    print("no adapter attached: run with --attach models/week1_run3/sft/adapter=qwen-sql-week1-adapter")

## 4. Capacity math first, then start the server and compare
Predict the KV-cache capacity from `config.json` and the measured weight size *before* vLLM reports its own number.

In [ ]:
if STATUS.get("merge") == "ok":
    weight_gib = sum(os.path.getsize(f) for f in glob.glob(f"{MODEL}/*.safetensors")) / 1024**3
    print(f"weights on disk: {weight_gib:.2f} GiB")
    run("kv_predict", f"python -m bench.kv_cache --model {MODEL} --weight-gib {weight_gib:.2f} --seq-len 2048 4096 16384")
# One GPU only: the single-GPU baseline every Week 3 config is compared against.
run("server", f"CUDA_VISIBLE_DEVICES=0 python -m bench.server --model {MODEL} --profile baseline --log {OUT}/server.log",
    needs=["merge"])
!grep -E "KV cache size|Maximum concurrency|Using .* backend|model weights took|Available KV" {OUT}/server.log | tee {OUT}/kv_reported.txt

## 5. Closed-loop sweeps (capacity curves)
Synthetic workloads use `ignore_eos` (fixed output length). Spider uses natural lengths: that's what production traffic does.

In [ ]:
for wl, levels in SWEEP.items():
    eos = "--no-ignore-eos" if wl.startswith("spider") else ""
    run(f"sweep_{wl}", f"python -m bench.client --url {URL} --model {SERVED} --prompts {WL}/{wl}.jsonl "
        f"--concurrency {' '.join(map(str, levels))} {eos} --label baseline/{wl} --out {OUT}/runs/{wl}",
        needs=["server"])

## 6. Open-loop Poisson arrivals on Spider
Rates are set as fractions of the closed-loop peak req/s. Near utilization 1, queueing makes TTFT explode; closed loop hides this, because the client waits for a response before sending the next request.

In [ ]:
import csv
if STATUS.get("sweep_spider_v1") == "ok":
    rows = list(csv.DictReader(open(f"{OUT}/runs/spider_v1/summary.csv")))
    peak = max(float(r["request_throughput_rps"]) for r in rows)
    rates = [round(f * peak, 2) for f in (0.25, 0.5, 0.75, 0.9, 1.0, 1.2)]
    print(f"closed-loop peak {peak:.2f} req/s -> rates {rates}")
    run("open_spider_v1", f"python -m bench.client --url {URL} --model {SERVED} --prompts {WL}/spider_v1.jsonl "
        f"--rate {' '.join(map(str, rates))} --no-ignore-eos --label baseline/spider_v1_open --out {OUT}/runs/spider_v1_open",
        needs=["server"])

## 7. Cross-check against `vllm bench serve` (random 128 in / 128 out, same as short_v1)
Agreement within ~10% on throughput and TPOT means our client isn't the bottleneck or mismeasuring.

In [ ]:
for c in (1, 8, 32):
    run(f"vllm_bench_c{c}", f"vllm bench serve --backend vllm --base-url {URL} --model {SERVED} --tokenizer {MODEL} "
        f"--dataset-name random --random-input-len 128 --random-output-len 128 --num-prompts 200 "
        f"--max-concurrency {c} --ignore-eos --percentile-metrics ttft,tpot,itl,e2el --metric-percentiles 50,95,99 "
        f"--save-result --result-dir {OUT}/vllm_bench --result-filename c{c}.json", needs=["server"])

## 8. Report, stop the server, collect results

In [ ]:
import shutil, signal
summaries = " ".join(glob.glob(f"{OUT}/runs/*/summary.csv"))
if summaries:
    run("report", f"python -m bench.report {summaries} --out {OUT}/report")
pidfile = f"{OUT}/server.pid"
if os.path.exists(pidfile):
    try:
        os.killpg(int(open(pidfile).read()), signal.SIGTERM)  # server runs in its own session
    except ProcessLookupError:
        pass
shutil.copy(f"{WL}/MANIFEST.json", OUT) if os.path.exists(f"{WL}/MANIFEST.json") else print("missing MANIFEST")
shutil.rmtree(WL, ignore_errors=True)  # regenerated deterministically; the manifest proves identity
json.dump(STATUS, open(f"{OUT}/status.json", "w"), indent=2)
print("step status:", STATUS)
shutil.make_archive("/kaggle/working/week2_results", "zip", OUT)
for d in ["/kaggle/working/data", MODEL]:  # keep kernel output small
    shutil.rmtree(d, ignore_errors=True)
print("download /kaggle/working/week2_results.zip")